# Experiment 5 — Reinforcement Learning

**Course:** CSAI4014 — Algorithms for Intelligent Systems and Robotics  
**Experiment:** Grid-world Markov decision process and tabular Q-learning

Fill in your name, roll number and batch in the lab record before submission.

## Aim and objectives
Formulate navigation as an MDP, implement tabular Q-learning, examine exploration-rate effects, and extract a greedy policy from the learned Q-table.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

H, W = 5, 7
WALLS = {(1,2), (2,2), (3,2), (1,4), (2,4)}
GOAL, PIT, START = (0, W-1), (2, W-1), (H-1, 0)
ACTIONS = [(-1,0), (1,0), (0,-1), (0,1)]  # up, down, left, right
ACTION_SYMBOLS = ['↑', '↓', '←', '→']
print('Grid:', H, 'x', W, '| start:', START, '| goal:', GOAL, '| pit:', PIT)

## 1. Define the environment
A wall collision leaves the agent in its current cell. Reaching the goal gives +10, reaching the pit gives −10, and each ordinary step costs −0.1.

In [ ]:
def step(state, action):
    r, c = state
    dr, dc = ACTIONS[action]
    nr, nc = r + dr, c + dc
    if not (0 <= nr < H and 0 <= nc < W) or (nr, nc) in WALLS:
        nr, nc = r, c
    next_state = (nr, nc)
    if next_state == GOAL:
        return next_state, 10.0, True
    if next_state == PIT:
        return next_state, -10.0, True
    return next_state, -0.1, False

# Check representative transitions
print('Ordinary step:', step(START, 0))
print('Goal transition:', step((0, W-2), 3))

## 2. Implement tabular Q-learning
The update uses the temporal-difference error. The future-value term is suppressed at terminal states.

In [ ]:
def train(eps, episodes=600, alpha=0.5, gamma=0.95, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((H, W, 4))
    curve = []
    for _ in range(episodes):
        state, total = START, 0.0
        for _ in range(120):
            if rng.random() < eps:
                action = int(rng.integers(4))
            else:
                action = int(np.argmax(Q[state[0], state[1]]))
            next_state, reward, done = step(state, action)
            total += reward
            Q[state[0], state[1], action] += alpha * (reward + gamma*np.max(Q[next_state[0], next_state[1]])*(not done) - Q[state[0], state[1], action])
            state = next_state
            if done:
                break
        curve.append(total)
    return Q, np.array(curve)

Q_demo, curve_demo = train(0.20)
print('Episodes:', len(curve_demo), '| final-100 mean return:', round(curve_demo[-100:].mean(), 3))

## 3. Compare exploration rates
Run 600 episodes for ε = 0.05, 0.20 and 0.50. The plot uses a 25-episode moving average.

In [ ]:
epsilons = (0.05, 0.20, 0.50)
results = {}
plt.figure(figsize=(9,4))
for eps in epsilons:
    Q, curve = train(eps, seed=0)
    results[eps] = (Q, curve)
    smoothed = np.convolve(curve, np.ones(25)/25, mode='valid')
    plt.plot(smoothed, label=f'epsilon = {eps}')
    print(f'epsilon={eps:.2f}: mean return over last 100 episodes = {curve[-100:].mean():.2f}')
plt.xlabel('episode'); plt.ylabel('return per episode (25-episode mean)')
plt.title('Q-learning exploration-rate comparison'); plt.legend(); plt.tight_layout(); plt.show()

## 4. Extract and display the greedy policy
The policy selects the action with the largest Q-value in each cell.

In [ ]:
Q, _ = results[0.20]
policy = np.argmax(Q, axis=2)
fig, ax = plt.subplots(figsize=(8,5))
ax.set_xlim(-0.5,W-0.5); ax.set_ylim(H-0.5,-0.5); ax.set_xticks(range(W)); ax.set_yticks(range(H)); ax.grid(True)
for r in range(H):
    for c in range(W):
        pos = (r,c)
        if pos in WALLS:
            ax.add_patch(plt.Rectangle((c-.5,r-.5),1,1,color='lightgray'))
        elif pos == GOAL:
            ax.add_patch(plt.Rectangle((c-.5,r-.5),1,1,color='palegreen')); ax.text(c,r,'+10',ha='center',va='center')
        elif pos == PIT:
            ax.add_patch(plt.Rectangle((c-.5,r-.5),1,1,color='salmon')); ax.text(c,r,'-10',ha='center',va='center')
        else:
            ax.text(c,r,ACTION_SYMBOLS[policy[r,c]],ha='center',va='center',fontsize=16)
ax.add_patch(plt.Rectangle((START[1]-.5,START[0]-.5),1,1,fill=False,edgecolor='blue',linewidth=2))
ax.text(START[1],START[0],'S',ha='center',va='center',color='blue',weight='bold')
ax.set_title('Greedy policy at epsilon = 0.20'); plt.tight_layout(); plt.show()
print('Policy action indices (0=up, 1=down, 2=left, 3=right):\\n', policy)

## Observations and submission checks
Record grid dimensions, reward structure, α=0.5, γ=0.95, 600 episodes, the final-100 mean return for each ε, and the extracted policy grid. Explain why excessive exploration lowers the return earned even when the greedy policy is useful.

**Viva focus:** Q-learning update terms; terminal-state masking; purpose of step cost; ε-greedy exploration; meaning of off-policy.